# PlagIA Enterprise · 03 — Entrenamiento en PyTorch (AMP FP16 en CUDA) + YOLO y Registro en MLflow

**Subsistema de Entrenamiento Profundo y MLOps (`ml_engine`)**

### Objetivos Técnicos
1. Ejecutar el entrenamiento acelerado por GPU (**NVIDIA RTX 3050 4 GB VRAM** con `torch.amp.autocast` FP16 y acumulación de gradientes).
2. Comparar **3 corridas de referencia** registradas en **MLflow (`sqlite:///mlflow.db`)**:
   - **Run 1:** `ResNet50` Fase A (Backbone congelado, `lr=1e-3`, `dropout=0.3`).
   - **Run 2:** `EfficientNet-B0` Fase A (Backbone congelado, `lr=1e-3`, `dropout=0.4`).
   - **Run 3:** `ResNet50` Fase B (**Fine-Tuning** de bloques profundos, `lr=1e-4`, `dropout=0.3`).
3. Probar el localizador fitosanitario **YOLO (`YOLOPhytosanitaryDetector`)** para extracción de cajas delimitadoras (*bounding boxes*).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import pandas as pd

from ml_engine.architectures import YOLOPhytosanitaryDetector, get_compute_device
from ml_engine.config import (
    BATCH_SIZE,
    DRONE_CLASSES,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_TRACKING_URI,
    SPLIT_MANIFEST_PATH,
)
from ml_engine.preprocessing import build_dataloaders, load_and_prepare_image
from ml_engine.trainer import train_and_log_mlflow

print("Dispositivo de cómputo activo:", get_compute_device())
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

split_df = pd.read_csv(SPLIT_MANIFEST_PATH)
loaders = build_dataloaders(split_df, batch_size=BATCH_SIZE)

In [ ]:
EPOCHS = 3

run_1 = train_and_log_mlflow(
    loaders["train"],
    loaders["val"],
    run_name="resnet50_frozen_lr1e-3",
    backbone_name="resnet50",
    class_names=DRONE_CLASSES,
    learning_rate=1e-3,
    dropout=0.3,
    epochs=EPOCHS,
    fine_tune=False,
    tags={"stage_type": "Phase_A_Frozen", "target_region": "Tolima"},
)

run_2 = train_and_log_mlflow(
    loaders["train"],
    loaders["val"],
    run_name="efficientnet_b0_frozen_lr1e-3",
    backbone_name="efficientnet_b0",
    class_names=DRONE_CLASSES,
    learning_rate=1e-3,
    dropout=0.4,
    epochs=EPOCHS,
    fine_tune=False,
    tags={"stage_type": "Phase_A_Frozen", "target_region": "Tolima"},
)

run_3 = train_and_log_mlflow(
    loaders["train"],
    loaders["val"],
    run_name="resnet50_finetuned_lr1e-4",
    backbone_name="resnet50",
    class_names=DRONE_CLASSES,
    learning_rate=1e-4,
    dropout=0.3,
    epochs=EPOCHS,
    fine_tune=True,
    tags={"stage_type": "Phase_B_FineTuning", "target_region": "Tolima"},
)

print("Resumen de corridas registradas en MLflow:")
for r in (run_1, run_2, run_3):
    print(r["run_name"], "-> F1-Macro:", round(r["metrics"]["f1_macro"], 4), "| run_id:", r["run_id"])

## 2. Localización de Focos Fitosanitarios con Detector Híbrido YOLO

In [ ]:
detector = YOLOPhytosanitaryDetector()
rust_row = split_df[split_df["label_name"] != "sano"].iloc[0]
img_rgb = load_and_prepare_image(rust_row["path"], apply_clahe=True)
boxes = detector.locate_lesions(img_rgb, predicted_class=rust_row["label_name"])
print(f"Focos localizados en '{rust_row['filename']}' ({rust_row['label_name']}):")
for b in boxes:
    print(" -", b)